In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class STN(nn.Module):
    def __init__(self):
        super(STN, self).__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(256, 8, kernel_size=7),
            nn.ReLU(True),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(8, 10, kernel_size=5),
            nn.ReLU(True),
            nn.MaxPool2d(2, stride=2)
        )

        self.fc_loc = nn.Sequential(
            nn.Linear(10 * 3 * 3, 32),
            nn.ReLU(True),
            nn.Linear(32, 3 * 2)
        )

        # Initialize the weights/bias with identity transformation
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, 10 * 3 * 3)
        theta = self.fc_loc(xs)
        theta = theta.view(-1, 2, 3)

        grid = F.affine_grid(theta, x.size())
        x = F.grid_sample(x, grid)

        return x


In [4]:
import torchvision
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

class MaskRCNNWithSTN(MaskRCNN):
    def __init__(self, backbone, num_classes):
        super(MaskRCNNWithSTN, self).__init__(backbone, num_classes)
        self.stn = STN()

    def forward(self, images, targets=None):
        # Apply STN before feeding images to the backbone
        transformed_images = [self.stn(image.unsqueeze(0)).squeeze(0) for image in images]
        
        return super(MaskRCNNWithSTN, self).forward(transformed_images, targets)

# Instantiate the model with STN
backbone = resnet_fpn_backbone('resnet50', pretrained=True)
model = MaskRCNNWithSTN(backbone, num_classes=2)  # 91 for COCO dataset


In [14]:
import os
import torch
import numpy as np
from torchvision.io import read_image
from torchvision.ops.boxes import masks_to_boxes
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, mask_dir, transforms=None):

        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms #torch.utils.data.Dataset
        self.imgs = [
            file for file in sorted(os.listdir(os.path.join(img_dir, "test_practice")))
            if file.lower().endswith(('.jpg'))
        ]
        self.masks = list(sorted(os.listdir(mask_dir)))

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "test_practice", self.imgs[idx])
        mask_path = os.path.join(self.root, self.masks[idx])
        img = read_image(img_path)
        mask = read_image(mask_path)
        obj_ids = torch.unique(mask)
        obj_ids = obj_ids[1:]
        num_objs = len(obj_ids)
        boxes = masks_to_boxes(masks)
        masks = (mask == obj_ids[:, None, None]).to(dtype=torch.uint8)
        labels = torch.ones((num_objs,), dtype=torch.int64)

        image_id = idx
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        # suppose all instances are not crowd
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        # Wrap sample and targets into torchvision tv_tensors:
        img = tv_tensors.Image(img)

        target = {}
        target["boxes"] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(img))
        target["masks"] = tv_tensors.Mask(masks)
        target["labels"] = labels
        target["image_id"] = image_id
        target["area"] = area
        target["iscrowd"] = iscrowd

        if self.transforms is not None:
            img, target = self.transforms(img, target)
        # Convert mask to tensor and encode it
        
        return img, target

    def __len__(self):
        return len(self.imgs)


In [15]:
from torch.utils.data import DataLoader
import torchvision.transforms as T
# Define transformations
transform = T.Compose([T.ToTensor()])

# Load your custom dataset
dataset = CustomDataset('test_practice', 'labelled_images', transforms=transform)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4, collate_fn=lambda x: tuple(zip(*x)))


: 

In [ ]:
num_epochs = 5
device = torch.cuda()
for epoch in range(num_epochs):
    model.train()
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {losses.item()}")
